# INSTRUCTOR SOLUTION: RL Fundamentals and MDPs
## AIAT 123 - Reinforcement Learning

**⚠️ INSTRUCTOR USE ONLY**

This solution demonstrates the complete implementation.


In [ ]:
import numpy as np

np.set_printoptions(precision=2, suppress=True)

N_ROWS, N_COLS = 3, 3
N_STATES = N_ROWS * N_COLS
ACTIONS = ["up", "right", "down", "left"]
ACTION_TO_DELTA = {
    0: (-1, 0),
    1: (0, 1),
    2: (1, 0),
    3: (0, -1),
}
ACTION_TO_ARROW = {0: "↑", 1: "→", 2: "↓", 3: "←"}
GOAL_STATE = 8
PIT_STATE = 6
TERMINAL_STATES = {GOAL_STATE, PIT_STATE}
GAMMA = 0.90


def to_pos(state):
    return divmod(state, N_COLS)


def to_state(row, col):
    return row * N_COLS + col


def transition(state, action):
    if state in TERMINAL_STATES:
        return state, 0.0

    row, col = to_pos(state)
    d_row, d_col = ACTION_TO_DELTA[action]
    next_row = min(max(row + d_row, 0), N_ROWS - 1)
    next_col = min(max(col + d_col, 0), N_COLS - 1)
    next_state = to_state(next_row, next_col)

    if next_state == GOAL_STATE:
        return next_state, 10.0
    if next_state == PIT_STATE:
        return next_state, -10.0
    return next_state, -1.0


def print_values(values):
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_state(r, c)
            if s == GOAL_STATE:
                row.append("  G   ")
            elif s == PIT_STATE:
                row.append("  P   ")
            else:
                row.append(f"{values[s]:6.2f}")
        print(" ".join(row))


def print_policy(policy):
    for r in range(N_ROWS):
        row = []
        for c in range(N_COLS):
            s = to_state(r, c)
            if s == GOAL_STATE:
                row.append(" G ")
            elif s == PIT_STATE:
                row.append(" P ")
            else:
                row.append(f" {ACTION_TO_ARROW[policy[s]]} ")
        print(" ".join(row))


print("✅ Setup complete!")
print("State layout: 0 1 2 / 3 4 5 / P 7 G")

In [ ]:
# Task 1: Define the MDP
print("Task 1: MDP Definition")
print("- States: integers 0 to 8 in a 3x3 grid")
print("- Actions: up, right, down, left")
print("- Terminal states: pit=6, goal=8")
print("- Reward: +10 for goal, -10 for pit, -1 otherwise")
print()


# Task 2: Value Iteration
def value_iteration(gamma=GAMMA, theta=1e-6):
    values = np.zeros(N_STATES)
    while True:
        delta = 0.0
        new_values = values.copy()
        for state in range(N_STATES):
            if state in TERMINAL_STATES:
                continue
            action_returns = []
            for action in range(len(ACTIONS)):
                next_state, reward = transition(state, action)
                action_returns.append(reward + gamma * values[next_state])
            best_value = max(action_returns)
            delta = max(delta, abs(best_value - values[state]))
            new_values[state] = best_value
        values = new_values
        if delta < theta:
            break
    return values


def greedy_policy_from_values(values, gamma=GAMMA):
    policy = np.zeros(N_STATES, dtype=int)
    for state in range(N_STATES):
        if state in TERMINAL_STATES:
            continue
        action_returns = []
        for action in range(len(ACTIONS)):
            next_state, reward = transition(state, action)
            action_returns.append(reward + gamma * values[next_state])
        policy[state] = int(np.argmax(action_returns))
    return policy


optimal_values = value_iteration()
optimal_policy = greedy_policy_from_values(optimal_values)

print("Task 2: Optimal values")
print_values(optimal_values)
print("\nTask 2: Greedy policy from optimal values")
print_policy(optimal_policy)
print()


# Task 3: Policy Evaluation and Comparison
def evaluate_policy(policy, gamma=GAMMA, theta=1e-6):
    values = np.zeros(N_STATES)
    while True:
        delta = 0.0
        for state in range(N_STATES):
            if state in TERMINAL_STATES:
                continue
            action = policy[state]
            next_state, reward = transition(state, action)
            new_value = reward + gamma * values[next_state]
            delta = max(delta, abs(new_value - values[state]))
            values[state] = new_value
        if delta < theta:
            break
    return values


right_then_down_policy = np.array([
    1, 1, 2,
    1, 1, 2,
    0, 1, 0,
])

safe_up_policy = np.array([
    1, 1, 2,
    0, 0, 2,
    0, 0, 0,
])

values_right_down = evaluate_policy(right_then_down_policy)
values_safe = evaluate_policy(safe_up_policy)

print("Task 3: Policy comparison")
print("\nPolicy A: right then down")
print_policy(right_then_down_policy)
print_values(values_right_down)

print("\nPolicy B: safer movement away from the pit")
print_policy(safe_up_policy)
print_values(values_safe)

print("\nInstructor note:")
print("The better policy is the one that reaches the goal while reliably avoiding the pit.")
print("Students should explain *why* the values differ, not just print the arrays.")